# Beyond Rank Fusion: A Practical Guide to Elasticsearch Retrievers

---
## Install Prerequisites

In [3]:
! pip install -q -U -r requirements.txt

---
## Provision Elastic Serverless (Terraform)

In [4]:
%%bash
terraform -chdir=terraform init -upgrade
terraform -chdir=terraform apply -auto-approve

Initializing the backend...
Initializing provider plugins...
- Finding elastic/elasticstack versions matching "~> 0.14"...
- Finding elastic/ec versions matching "~> 0.12"...
- Using previously-installed elastic/elasticstack v0.14.3
- Using previously-installed elastic/ec v0.12.4

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # ec_elasticsearch_project.demo_project will be created
  + resource "ec_elastics

---
## Create .env from Terraform Outputs

In [5]:
%%bash
cat > .env << EOF
ELASTIC_USERNAME=$(terraform -chdir=terraform output -raw elastic_username)
ELASTIC_PASSWORD=$(terraform -chdir=terraform output -raw elastic_password)
ELASTIC_CLOUD_ID=$(terraform -chdir=terraform output -raw elastic_cloud_id)
EOF

---
## Environment Setup

In [6]:
import json
import os

from dotenv import load_dotenv
from elasticsearch import Elasticsearch
import pandas as pd

from lib.helpers import create_index, ingest_products, display_results, side_by_side

load_dotenv(override=True)

client = Elasticsearch(
    cloud_id=os.environ["ELASTIC_CLOUD_ID"],
    basic_auth=(os.environ["ELASTIC_USERNAME"], os.environ["ELASTIC_PASSWORD"]),
)

with open("data/products.json") as f:
    products = json.load(f)

print(f"Loaded {len(products)} products")

create_index(client)
ingest_products(client, products)
client.indices.refresh(index="hand_tools")
count = client.count(index="hand_tools")
print(f"Document count: {count['count']}")

Loaded 80 products
Created index 'hand_tools'
Indexed 80 documents into 'hand_tools'
Document count: 80


---
## Baseline: Lexical vs Semantic

In [7]:
# BM25 keyword search
QUERY = "versatile fastening tool for woodworking joints"

bm25_resp = client.search(
    index="hand_tools",
    body={
        "query": {
            "multi_match": {
                "query": QUERY,
                "fields": ["name", "description"],
            }
        },
        "size": 5,
    },
)

bm25_df = display_results(bm25_resp["hits"]["hits"])
print("BM25 Keyword Search Results:")
display(bm25_df)

# Semantic search via EIS
semantic_resp = client.search(
    index="hand_tools",
    body={
        "query": {
            "semantic": {
                "field": "description_semantic",
                "query": QUERY,
            }
        },
        "size": 5,
    },
)

semantic_df = display_results(semantic_resp["hits"]["hits"])
print("\nSemantic Search Results:")
display(semantic_df)

BM25 Keyword Search Results:


,_score,name,avg_rating,units_sold_30d,price
0,4.734287,Stanley No.5 Jack Plane,4.2,135,69.99
1,3.671693,Zona 35-670 Ultra Thin Kerf Razor Saw,4.1,120,8.99
2,2.810300,Milwaukee 48-22-2761 11-in-1 Multi-Tip Driver,4.3,490,17.99
3,2.633790,Veritas Dovetail Saw 20 TPI,4.8,65,84.99
4,2.436905,Schroeder Hand Drill with Side Handle,4.0,35,28.50



Semantic Search Results:


,_score,name,avg_rating,units_sold_30d,price
0,0.836517,Record 52-1/2 Quick-Release Woodworking Vise,4.6,210,139.00
1,0.801652,Milwaukee 48-22-2761 11-in-1 Multi-Tip Driver,4.3,490,17.99
2,0.798405,Jorgensen 3712-HD 12-Inch Wood Handscrew Clamp,4.5,185,26.99
3,0.793927,Gyokucho 770-3600 Dozuki Fine Crosscut Saw,4.7,145,39.99
4,0.793439,Thor 712R Copper and Rawhide Mallet,4.4,85,38.50


---
## RRF Retriever (Reciprocal Rank Fusion)

In [8]:
# Unweighted RRF
QUERY = "precision screwdriver for electronics repair"

rrf_resp = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        }
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        }
                    },
                ],
                "rank_window_size": 50,
            }
        },

        "size": 7,
    },
)

rrf_df = display_results(rrf_resp["hits"]["hits"])
rrf_df

,_score,name,avg_rating,units_sold_30d,price
0,0.032787,PrecisionDrive 6-Piece Screwdriver Set,3.6,95,34.99
1,0.032258,Wiha Precision Micro Screwdriver Set,4.6,280,42.99
2,0.030777,Tekton Everybit Ratchet Driver and Bit Set,4.1,200,28.99
3,0.030366,Wera Kraftform Big Pack 300 Screwdriver Set,4.4,350,59.99
4,0.030077,Vessel MegaDora Impacta No.2 Phillips Driver,4.5,265,14.99
5,0.029631,Knipex 2611200 Long Nose Pliers 8-Inch,4.7,215,29.99
6,0.029514,Mitutoyo 530-312 Vernier Caliper 6-Inch,4.8,78,44.99


In [9]:
# Weighted RRF — boost BM25 retriever (weight: 2.0)
rrf_bm25_boost = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        },
                        "weight": 2.0,
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        }
                    }
                ],
                "rank_constant": 60,
                "rank_window_size": 50,
            }
        },
        "size": 7,
    },
)

rrf_bm25_df = display_results(rrf_bm25_boost["hits"]["hits"])
rrf_bm25_df

,_score,name,avg_rating,units_sold_30d,price
0,0.049180,PrecisionDrive 6-Piece Screwdriver Set,3.6,95,34.99
1,0.048387,Wiha Precision Micro Screwdriver Set,4.6,280,42.99
2,0.045928,Tekton Everybit Ratchet Driver and Bit Set,4.1,200,28.99
3,0.045139,Mitutoyo 530-312 Vernier Caliper 6-Inch,4.8,78,44.99
4,0.045002,Vessel MegaDora Impacta No.2 Phillips Driver,4.5,265,14.99
5,0.044859,Wera Kraftform Big Pack 300 Screwdriver Set,4.4,350,59.99
6,0.044337,Knipex 2611200 Long Nose Pliers 8-Inch,4.7,215,29.99


In [10]:
# Weighted RRF — boost semantic retriever (weight: 2.0)
rrf_sem_boost = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        }      
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        },
                        "weight": 2.0,
                    }
                ],
                "rank_constant": 60,
                "rank_window_size": 50,
            }
        },
        "size": 7,
    },
)

rrf_sem_df = display_results(rrf_sem_boost["hits"]["hits"])
rrf_sem_df

,_score,name,avg_rating,units_sold_30d,price
0,0.049180,PrecisionDrive 6-Piece Screwdriver Set,3.6,95,34.99
1,0.048387,Wiha Precision Micro Screwdriver Set,4.6,280,42.99
2,0.046402,Tekton Everybit Ratchet Driver and Bit Set,4.1,200,28.99
3,0.046239,Wera Kraftform Big Pack 300 Screwdriver Set,4.4,350,59.99
4,0.045228,Vessel MegaDora Impacta No.2 Phillips Driver,4.5,265,14.99
5,0.044557,Knipex 2611200 Long Nose Pliers 8-Inch,4.7,215,29.99
6,0.044283,Vessel Impacta No.2 JIS Driver,4.8,410,18.50


In [11]:
# Side-by-side: unweighted vs BM25-boosted vs semantic-boosted
side_by_side({"rrf": rrf_df, "rrf_bm25_boost": rrf_bm25_df, "rrf_sem_boost": rrf_sem_df})

,name,rrf_score,rrf_bm25_boost_score,rrf_sem_boost_score
0,PrecisionDrive 6-Piece Screwdriver Set,0.032787,0.04918,0.04918
1,Wiha Precision Micro Screwdriver Set,0.032258,0.048387,0.048387
2,Tekton Everybit Ratchet Driver and Bit Set,0.030777,0.045928,0.046402
3,Wera Kraftform Big Pack 300 Screwdriver Set,0.030366,0.044859,0.046239
4,Vessel MegaDora Impacta No.2 Phillips Driver,0.030077,0.045002,0.045228
5,Knipex 2611200 Long Nose Pliers 8-Inch,0.029631,0.044337,0.044557
6,Mitutoyo 530-312 Vernier Caliper 6-Inch,0.029514,0.045139,—
7,Vessel Impacta No.2 JIS Driver,—,—,0.044283


---
## Linear Retriever

In [12]:
# RRF baseline for this query
QUERY = "mortise chisel for hand-cut joinery"

rrf_baseline = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rrf": {
                "retrievers": [
                    {
                        "standard": {
                            "query": {
                                "multi_match": {
                                    "query": QUERY,
                                    "fields": ["name", "description"],
                                }
                            }
                        }
                    },
                    {
                        "standard": {
                            "query": {
                                "semantic": {
                                    "field": "description_semantic",
                                    "query": QUERY,
                                }
                            }
                        }
                    },
                ],
                "rank_constant": 60,
                "rank_window_size": 50,
            }
        },
        "size": 7,
    },
)

rrf_base_df = display_results(rrf_baseline["hits"]["hits"])
rrf_base_df

,_score,name,avg_rating,units_sold_30d,price
0,0.032787,Mortise Chisel Heavy-Duty Hand Cut Joinery Chisel,3.8,140,42.00
1,0.032002,Narex Premium Bench Chisel 3/4-Inch,4.3,195,28.00
2,0.031258,Record 52-1/2 Quick-Release Woodworking Vise,4.6,210,139.00
3,0.030777,Faithfull FAIJWM Joiners Wooden Mallet,3.9,165,12.99
4,0.030777,Gyokucho 770-3600 Dozuki Fine Crosscut Saw,4.7,145,39.99
5,0.029851,Two Cherries 1/2-Inch Firmer Chisel,4.4,105,32.00
6,0.029670,Schroeder Hand Drill with Side Handle,4.0,35,28.50


In [13]:
# Linear retriever — MinMax normalization, BM25 0.3 / semantic 0.7 
linear_swap = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "linear": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        },
                        "weight": 0.3,
                        "normalizer": "minmax",
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        },
                        "weight": 0.7,
                        "normalizer": "minmax",
                    },
                ],
            }
        },
        "size": 7,
    },
)

linear_swap_df = display_results(linear_swap["hits"]["hits"])
linear_swap_df

,_score,name,avg_rating,units_sold_30d,price
0,1.000000,Mortise Chisel Heavy-Duty Hand Cut Joinery Chisel,3.8,140,42.00
1,0.303437,Narex Premium Bench Chisel 3/4-Inch,4.3,195,28.00
2,0.213128,Record 52-1/2 Quick-Release Woodworking Vise,4.6,210,139.00
3,0.187110,Faithfull FAIJWM Joiners Wooden Mallet,3.9,165,12.99
4,0.099665,Schroeder Hand Drill with Side Handle,4.0,35,28.50
5,0.098452,"Two Cherries 3/4"" Pigsticker Oval-Bolster Mort...",4.9,88,94.00
6,0.085372,Gyokucho 770-3600 Dozuki Fine Crosscut Saw,4.7,145,39.99


In [14]:
# Linear retriever — MinMax normalization, BM25 0.6 / semantic 0.4
linear_minmax = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "linear": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        },
                        "weight": 0.6,
                        "normalizer": "minmax",
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        },
                        "weight": 0.4,
                        "normalizer": "minmax",
                    },
                ],
            }
        },
        "size": 7,
    },
)

linear_mm_df = display_results(linear_minmax["hits"]["hits"])
linear_mm_df

,_score,name,avg_rating,units_sold_30d,price
0,1.000000,Mortise Chisel Heavy-Duty Hand Cut Joinery Chisel,3.8,140,42.00
1,0.266458,Narex Premium Bench Chisel 3/4-Inch,4.3,195,28.00
2,0.196903,"Two Cherries 3/4"" Pigsticker Oval-Bolster Mort...",4.9,88,94.00
3,0.157993,Record 52-1/2 Quick-Release Woodworking Vise,4.6,210,139.00
4,0.140532,Faithfull FAIJWM Joiners Wooden Mallet,3.9,165,12.99
5,0.103982,Gyokucho 770-3600 Dozuki Fine Crosscut Saw,4.7,145,39.99
6,0.056951,Schroeder Hand Drill with Side Handle,4.0,35,28.50


In [15]:
# Side-by-side: RRF vs Linear (BM25-heavy) vs Linear (semantic-heavy)
side_by_side({"rrf": rrf_base_df, "linear_0.6_bm25": linear_mm_df, "linear_0.7_sem": linear_swap_df})

,name,rrf_score,linear_0.6_bm25_score,linear_0.7_sem_score
0,Mortise Chisel Heavy-Duty Hand Cut Joinery Chisel,0.032787,1.0,1.0
1,Narex Premium Bench Chisel 3/4-Inch,0.032002,0.266458,0.303437
2,Record 52-1/2 Quick-Release Woodworking Vise,0.031258,0.157993,0.213128
3,Gyokucho 770-3600 Dozuki Fine Crosscut Saw,0.030777,0.103982,0.085372
4,Faithfull FAIJWM Joiners Wooden Mallet,0.030777,0.140532,0.18711
5,Two Cherries 1/2-Inch Firmer Chisel,0.029851,—,—
6,Schroeder Hand Drill with Side Handle,0.02967,0.056951,0.099665
7,"Two Cherries 3/4"" Pigsticker Oval-Bolster Mort...",—,0.196903,0.098452


---
## Rescorer Retriever

In [23]:
# Linear baseline (no rescoring)
QUERY = "reliable claw hammer for general carpentry"

linear_base = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "linear": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        },
                        "weight": 0.6,
                        "normalizer": "minmax",
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        },
                        "weight": 0.4,
                        "normalizer": "minmax",
                    },
                ],
            }
        },
        "size": 7,
    },
)

linear_base_df = display_results(linear_base["hits"]["hits"])
linear_base_df

ConnectionError: Connection error caused by: ConnectionError(Connection error caused by: NameResolutionError(HTTPSConnection(host='ee7f18dffe074d90891c39b0db7c7633.es.us-central1.gcp.elastic.cloud', port=443): Failed to resolve 'ee7f18dffe074d90891c39b0db7c7633.es.us-central1.gcp.elastic.cloud' ([Errno -2] Name or service not known)))

In [ ]:
# Rescorer — boost by sales velocity and rating
rescorer_resp = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rescorer": {
                "retriever": {
                    "linear": {
                        "rank_window_size": 50,
                        "retrievers": [
                            {
                                "retriever": {
                                    "standard": {
                                        "query": {
                                            "multi_match": {
                                                "query": QUERY,
                                                "fields": ["name", "description"],
                                            }
                                        }
                                    }
                                },
                                "weight": 0.6,
                                "normalizer": "minmax",
                            },
                            {
                                "retriever": {
                                    "standard": {
                                        "query": {
                                            "semantic": {
                                                "field": "description_semantic",
                                                "query": QUERY,
                                            }
                                        }
                                    }
                                },
                                "weight": 0.4,
                                "normalizer": "minmax",
                            },
                        ],
                    }
                },
                "rescore": {
                    "window_size": 50,
                    "query": {
                        "rescore_query": {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    "source": "_score * Math.log1p(doc['units_sold_30d'].value) * (doc['avg_rating'].value / 5.0)"
                                },
                            }
                        },
                        "query_weight": 1,
                        "rescore_query_weight": 2,
                    },
                },
            }
        },
        "size": 7,
    },
)

rescorer_df = display_results(rescorer_resp["hits"]["hits"])
rescorer_df

,_score,name,avg_rating,units_sold_30d,price
0,14.881354,Estwing E3-16C 16oz Curved Claw Hammer,4.9,1840,37.99
1,11.870536,Lie-Nielsen Warrington Pattern Hammer,4.8,310,118.00
2,11.419826,Knipex 8701250 Cobra 10-Inch Water Pump Pliers,4.8,380,34.99
3,11.197245,Klein Tools 85076 7-Piece Cushion-Grip Screwdr...,4.6,420,39.99
4,10.860146,Irwin Quick-Grip XP600 12-Inch Bar Clamp,4.4,450,24.99
5,10.855662,Wera Kraftform Kompakt 25 Bit-Holding Screwdriver,4.7,310,34.50
6,10.701912,Stanley FatMax 20oz Framing Hammer,4.2,520,44.99


In [ ]:
# Rescorer + in_stock penalty
rescorer_stock = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "rescorer": {
                "retriever": {
                    "linear": {
                        "rank_window_size": 50,
                        "retrievers": [
                            {
                                "retriever": {
                                    "standard": {
                                        "query": {
                                            "multi_match": {
                                                "query": QUERY,
                                                "fields": ["name", "description"],
                                            }
                                        }
                                    }
                                },
                                "weight": 0.6,
                                "normalizer": "minmax",
                            },
                            {
                                "retriever": {
                                    "standard": {
                                        "query": {
                                            "semantic": {
                                                "field": "description_semantic",
                                                "query": QUERY,
                                            }
                                        }
                                    }
                                },
                                "weight": 0.4,
                                "normalizer": "minmax",
                            },
                        ],
                    }
                },
                "rescore": {
                    "window_size": 50,
                    "query": {
                        "rescore_query": {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    "source": "_score * Math.log1p(doc['units_sold_30d'].value) * (doc['avg_rating'].value / 5.0) * (doc['in_stock'].value ? 1.0 : 0.1)"
                                },
                            }
                        },
                        "query_weight": 1,
                        "rescore_query_weight": 2,
                    },
                },
            }
        },
        "size": 7,
    },
)

rescorer_stock_df = display_results(rescorer_stock["hits"]["hits"])
rescorer_stock_df

,_score,name,avg_rating,units_sold_30d,price
0,14.881354,Estwing E3-16C 16oz Curved Claw Hammer,4.9,1840,37.99
1,11.419826,Knipex 8701250 Cobra 10-Inch Water Pump Pliers,4.8,380,34.99
2,11.197245,Klein Tools 85076 7-Piece Cushion-Grip Screwdr...,4.6,420,39.99
3,10.860146,Irwin Quick-Grip XP600 12-Inch Bar Clamp,4.4,450,24.99
4,10.855662,Wera Kraftform Kompakt 25 Bit-Holding Screwdriver,4.7,310,34.50
5,10.701912,Stanley FatMax 20oz Framing Hammer,4.2,520,44.99
6,10.542920,DeWalt DWHT51054 20oz Rip Claw Hammer,4.5,295,29.99


In [ ]:
# Score breakdown — show the math behind rescoring
# final = query_weight * linear_score + rescore_query_weight * rescore_score
# With query_weight=1, rescore_query_weight=2:
#   final = linear_score + 2 * (linear_score * log1p(units_sold_30d) * avg_rating/5)

breakdown_resp = client.search(
    index="hand_tools",
    body={
        "retriever": {
            "linear": {
                "retrievers": [
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": QUERY,
                                        "fields": ["name", "description"],
                                    }
                                }
                            }
                        },
                        "weight": 0.6,
                        "normalizer": "minmax",
                    },
                    {
                        "retriever": {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "description_semantic",
                                        "query": QUERY,
                                    }
                                }
                            }
                        },
                        "weight": 0.4,
                        "normalizer": "minmax",
                    },
                ],
            }
        },
        "size": 7,
    },
)

import math
rows = []
for hit in breakdown_resp["hits"]["hits"]:
    src = hit["_source"]
    linear_score = hit["_score"]
    rating = src["avg_rating"]
    sold = src["units_sold_30d"]
    in_stock = src["in_stock"]

    rating_mult = rating / 5.0
    velocity_mult = math.log1p(sold)
    stock_mult = 1.0 if in_stock else 0.1
    rescore_score = linear_score * velocity_mult * rating_mult
    final = linear_score + 2 * rescore_score
    final_with_stock = linear_score + 2 * (rescore_score * stock_mult)

    rows.append({
        "name": src["name"],
        "linear_score": round(linear_score, 4),
        "avg_rating": rating,
        "units_sold_30d": sold,
        "in_stock": in_stock,
        "rating_mult": round(rating_mult, 2),
        "velocity_mult": round(velocity_mult, 2),
        "stock_mult": stock_mult,
        "rescored": round(final, 4),
        "rescored_stock": round(final_with_stock, 4),
    })

pd.DataFrame(rows)

,name,linear_score,avg_rating,units_sold_30d,in_stock,rating_mult,velocity_mult,stock_mult,rescored,rescored_stock
0,Reliable Claw Hammer General Carpentry 16oz,0.9791,1.9,14,True,0.38,2.71,1.0,2.9941,2.9941
1,Lie-Nielsen Warrington Pattern Hammer,0.8356,4.8,310,False,0.96,5.74,0.1,10.0442,1.7565
2,Vaughan 999L 20oz Framing Rip Hammer,0.2901,4.0,245,True,0.80,5.51,1.0,2.8454,2.8454
3,DeWalt DWHT51054 20oz Rip Claw Hammer,0.1831,4.5,295,True,0.90,5.69,1.0,2.0589,2.0589
4,Two Cherries 1/2-Inch Firmer Chisel,0.0830,4.4,105,True,0.88,4.66,1.0,0.7642,0.7642
5,Stanley 51-621 16oz Curve Claw Hammer,0.0692,4.3,380,True,0.86,5.94,1.0,0.7766,0.7766
6,Estwing E3-16C 16oz Curved Claw Hammer,0.0692,4.9,1840,True,0.98,7.52,1.0,1.0890,1.0890


In [ ]:
# Side-by-side: Linear vs Rescorer vs Rescorer+stock
side_by_side({"linear": linear_base_df, "rescorer": rescorer_df, "rescorer_stock": rescorer_stock_df})

,name,linear_score,rescorer_score,rescorer_stock_score
0,Reliable Claw Hammer General Carpentry 16oz,0.979079,—,—
1,Lie-Nielsen Warrington Pattern Hammer,0.835595,11.870536,—
2,Vaughan 999L 20oz Framing Rip Hammer,0.290091,—,—
3,DeWalt DWHT51054 20oz Rip Claw Hammer,0.183131,—,10.54292
4,Two Cherries 1/2-Inch Firmer Chisel,0.082991,—,—
5,Estwing E3-16C 16oz Curved Claw Hammer,0.069209,14.881354,14.881354
6,Stanley 51-621 16oz Curve Claw Hammer,0.069209,—,—
7,Irwin Quick-Grip XP600 12-Inch Bar Clamp,—,10.860146,10.860146
8,Klein Tools 85076 7-Piece Cushion-Grip Screwdr...,—,11.197245,11.197245
9,Knipex 8701250 Cobra 10-Inch Water Pump Pliers,—,11.419826,11.419826


---
## Summary & Comparison

In [21]:
# All four retrievers on a single query
QUERY = "durable hand tool for precise woodworking"

# 1. RRF (unweighted)
r1 = client.search(index="hand_tools", body={
    "retriever": {"rrf": {"retrievers": [
        {"standard": {"query": {"multi_match": {"query": QUERY, "fields": ["name", "description"]}}}},
        {"standard": {"query": {"semantic": {"field": "description_semantic", "query": QUERY}}}},
    ], "rank_constant": 60, "rank_window_size": 50}}, "size": 5})

# 2. RRF (weighted — BM25 boosted)
r2 = client.search(index="hand_tools", body={
    "retriever": {"rrf": {"retrievers": [
        {"retriever": {"standard": {"query": {"multi_match": {"query": QUERY, "fields": ["name", "description"]}}}}, "weight": 2.0},
        {"retriever": {"standard": {"query": {"semantic": {"field": "description_semantic", "query": QUERY}}}}},
    ], "rank_constant": 60, "rank_window_size": 50}}, "size": 5})

# 3. Linear + MinMax
r3 = client.search(index="hand_tools", body={
    "retriever": {"linear": {"retrievers": [
        {"retriever": {"standard": {"query": {"multi_match": {"query": QUERY, "fields": ["name", "description"]}}}}, "weight": 0.6, "normalizer": "minmax"},
        {"retriever": {"standard": {"query": {"semantic": {"field": "description_semantic", "query": QUERY}}}}, "weight": 0.4, "normalizer": "minmax"},
    ]}}, "size": 5})

# 4. Rescorer (relevance + business signals, same candidate pool as Linear)
r4 = client.search(index="hand_tools", body={
    "retriever": {"rescorer": {
        "retriever": {"linear": {"retrievers": [
            {"retriever": {"standard": {"query": {"multi_match": {"query": QUERY, "fields": ["name", "description"]}}}}, "weight": 0.6, "normalizer": "minmax"},
            {"retriever": {"standard": {"query": {"semantic": {"field": "description_semantic", "query": QUERY}}}}, "weight": 0.4, "normalizer": "minmax"},
        ], "rank_window_size": 5}},
        "rescore": {"window_size": 5, "query": {
            "rescore_query": {"script_score": {"query": {"match_all": {}},
                "script": {"source": "_score * Math.log1p(doc['units_sold_30d'].value) * (doc['avg_rating'].value / 5.0)"}}},
            "query_weight": 1, "rescore_query_weight": 2}},
    }}, "size": 5})

frames = {
    "rrf": display_results(r1["hits"]["hits"]),
    "rrf_weighted": display_results(r2["hits"]["hits"]),
    "linear_minmax": display_results(r3["hits"]["hits"]),
    "rescorer": display_results(r4["hits"]["hits"]),
}

side_by_side(frames)

,name,rrf_score,rrf_weighted_score,linear_minmax_score,rescorer_score
0,Gyokucho 770-3600 Dozuki Fine Crosscut Saw,0.032522,0.048916,0.861733,10.192046
1,Zona 35-670 Ultra Thin Kerf Razor Saw,0.031498,0.047123,0.653624,8.339065
2,Record 52-1/2 Quick-Release Woodworking Vise,0.030886,0.045379,0.400324,10.247418
3,Mortise Chisel Heavy-Duty Hand Cut Joinery Chisel,0.030777,0.045928,—,—
4,Schroeder Hand Drill with Side Handle,0.030622,0.046751,0.551879,6.207487
5,PrecisionDrive 6-Piece Screwdriver Set,—,—,0.465081,6.927618


---
## Teardown — Destroy Elastic Environment

In [22]:
%%bash
terraform -chdir=terraform destroy -auto-approve
rm -f .env

ec_elasticsearch_project.demo_project: Refreshing state... [id=ee7f18dffe074d90891c39b0db7c7633]

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  - destroy

Terraform will perform the following actions:

  # ec_elasticsearch_project.demo_project will be destroyed
  - resource "ec_elasticsearch_project" "demo_project" {
      - alias         = "demoproject" -> null
      - cloud_id      = "demo_project:dXMtY2VudHJhbDEuZ2NwLmVsYXN0aWMuY2xvdWQkZWU3ZjE4ZGZmZTA3NGQ5MDg5MWMzOWIwZGI3Yzc2MzMuZXMkZWU3ZjE4ZGZmZTA3NGQ5MDg5MWMzOWIwZGI3Yzc2MzMua2I=" -> null
      - credentials   = {
          - password = (sensitive value) -> null
          - username = "admin" -> null
        } -> null
      - endpoints     = {
          - elasticsearch = "https://demoproject-ee7f18.es.us-central1.gcp.elastic.cloud" -> null
          - kibana        = "https://demoproject-ee7f18.kb.us-central1.gcp.elastic.cloud" -> null
  